# Import Relevant Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from sklearn.linear_model import LinearRegression as lm
from sklearn.neighbors import KNeighborsRegressor as knn
from sklearn.tree import DecisionTreeRegressor as dt
from sklearn.ensemble import RandomForestRegressor as rf, GradientBoostingRegressor as gbf
from sklearn.svm import SVR as svr
from sklearn.neural_network import MLPRegressor as mlp

from sklearn.metrics import max_error as me, mean_absolute_error as mae, r2_score, mean_squared_error as mse # ,mean_absolute_percentage_error as mape
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings("ignore")

# Easy Evaluation Class

In [ ]:
def cust_mape(actual, pred): 
    # sklearn mape is broken
    data = pd.DataFrame({'act':actual,'pred':pred})
    data = data[data['act'] != 0]
    return np.mean(np.abs((data['act'] - data['pred']) / data['act']))

def evaluate_regression(tr_act, tr_pred, te_act = None, te_pred = None):
    """
    Arguments:
        1) tr_act (pandas series/numpy array): training actual target values
        2) tr_pred (pandas series/numpy array): training predicted target values
        3) te_act (pandas series/numpy array): testing actual target values
        4) te_pred (pandas series/numpy array): testing predicted target values

    Description:
        Generates a statistical summary of predictions vs actuals for a regression problem
        
    Returns:
        1) pandas dataframe

    Example:
        X = pd.DataFrame({'testvar':[2,1,4,3,4,5,6,7,8,9,14], 'var2':[0,1,2,3,4,5,6,7,8,9,10]})
        X2 = pd.DataFrame({'testvar':[4,4,4,4,4,9,9,9,9,9,9], 'var2':[0,1,2,3,4,5,6,7,8,9,10]})
        evaluate_regression(X['testvar'], X2['testvar'], X['var2'], X2['var2'])    
    
    """
    
    pd.options.display.float_format = '{:.4f}'.format
    
    metrics = ['Correl','R^2', 'MAE', 'RMSE', 'MAPE', 'Max Err']
    tr_stats = []
    
    # Training set
        # Correl
    rho, pval = pearsonr(tr_act, tr_pred)
    tr_stats.append(rho)
        # R^2
    tr_stats.append(r2_score(tr_act, tr_pred))
        # MAE
    tr_stats.append(mae(tr_act, tr_pred))
        # RMSE
    tr_stats.append(mse(tr_act, tr_pred)**.5)
        # MAPE
    tr_stats.append(cust_mape(tr_act, tr_pred))
        # Max Error
    tr_stats.append(me(tr_act, tr_pred))
    data = pd.DataFrame({'metric':metrics,'train':tr_stats})
    
    # Test Set
    if type(te_act) != type(None):
        te_stats = []
        rho, pval = pearsonr(te_act, te_pred)
        te_stats.append(rho)
            # R^2
        te_stats.append(r2_score(te_act, te_pred))
            # MAE
        te_stats.append(mae(te_act, te_pred))
            # RMSE
        te_stats.append(mse(te_act, te_pred)**.5)
            # MAPE
        te_stats.append(cust_mape(te_act, te_pred))
            # Max Error
        te_stats.append(me(te_act, te_pred))   
        data['test'] = te_stats
        
    return data

# Data Preprocessing

In [ ]:
# Import data set
path = r"C:\Users\charl\Downloads\FINA 4075 AI\cars_ai_practice.csv"
data = pd.read_csv(path)

# Alphebetize the columns
data.sort_index(axis=1, inplace=True)
data.head()

### Train Test Split

In [ ]:
# Split target from data
target = 'price'

# Separate target from the rest of the data
cols = list(data.columns)
cols.remove(target)

# Define dependent and independent variables
y = data[target]
X = data[cols]

# Immediately train test split
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size = 0.3, random_state = 42)

### Understand target variable

In [ ]:
y_tr.describe()

# Test Models

### Linear Regression

In [ ]:
# Instantiate    
    # No Hyperparameters for this model
lm_model = lm()

# Train the model 
lm_model.fit(x_tr, y_tr)

# Generate Predictions
lm_tr = lm_model.predict(x_tr)
lm_te = lm_model.predict(x_te)

# Evaluate
evaluate_regression(y_tr, lm_tr, y_te, lm_te)

### K-Nearest Neighbors

In [ ]:
# Instantiate    
    # Hyperparameters:
        # n_neighbors: integer
        # weights: set string {'uniform', 'distance'}
knn_model = knn(n_neighbors = 5, weights = 'uniform')

    # KNN requires the independent variables to be scaled
scaler = StandardScaler()
scaler.fit(x_tr)
x_tr_sc = scaler.transform(x_tr)
x_te_sc = scaler.transform(x_te)

# Train the model 
knn_model.fit(x_tr_sc, y_tr)

# Generate Predictions
knn_tr = knn_model.predict(x_tr_sc)
knn_te = knn_model.predict(x_te_sc)

# Evaluate
evaluate_regression(y_tr, knn_tr, y_te, knn_te)

### Decision Tree

In [ ]:
# Instantiate    
    # Hyperparameters:
        # max_depth: integer
        # max_features: set string {“auto”, “sqrt”, “log2”} or None
dt_model = dt(max_depth = 2, max_features = 'sqrt')

# Train the model 
dt_model.fit(x_tr, y_tr)

# Generate Predictions
dt_tr = dt_model.predict(x_tr)
dt_te = dt_model.predict(x_te)

# Evaluate
evaluate_regression(y_tr, dt_tr, y_te, dt_te)

### Random Forest

In [ ]:
# Instantiate    
    # Hyperparameters:
        # n_estimators: integer
        # max_depth: integer
        # max_features: set string {“auto”, “sqrt”, “log2”} or None
rf_model = rf(n_estimators = 150, max_depth = 4, max_features = 'sqrt')

# Train the model 
rf_model.fit(x_tr, y_tr)

# Generate Predictions
rf_tr = rf_model.predict(x_tr)
rf_te = rf_model.predict(x_te)

# Evaluate
evaluate_regression(y_tr, rf_tr, y_te, rf_te)

### Gradient Boosted Forest

In [ ]:
# Instantiate    
    # Hyperparameters:
        # n_estimators: integer
        # max_depth: integer
        # max_features: set string {“auto”, “sqrt”, “log2”} or None
gbf_model = gbf(n_estimators = 100, max_depth = 4, max_features = 'sqrt')

# Train the model 
gbf_model.fit(x_tr, y_tr)

# Generate Predictions
gbf_tr = gbf_model.predict(x_tr)
gbf_te = gbf_model.predict(x_te)

# Evaluate
evaluate_regression(y_tr, gbf_tr, y_te, gbf_te)

### Support Vector Regressor

In [ ]:
# Instantiate    
    # Hyperparameters:
        # kernel: set string {‘linear’, ‘poly’, ‘rbf’, ‘sigmoid’, ‘precomputed’}
        # C: float
svr_model = svr(kernel = 'sigmoid', C = 150)

    # SVR requires the independent variables to be scaled
scaler = StandardScaler()
scaler.fit(x_tr)
x_tr_sc = scaler.transform(x_tr)
x_te_sc = scaler.transform(x_te)

# Train the model 
svr_model.fit(x_tr_sc, y_tr)

# Generate Predictions
svr_tr = svr_model.predict(x_tr_sc)
svr_te = svr_model.predict(x_te_sc)

# Evaluate
evaluate_regression(y_tr, svr_tr, y_te, svr_te)

### Artificial/Feedforward Neural Network (aka: multilayer perceptron)

In [ ]:
# Instantiate    
    # Hyperparameters:
        # hidden_layer_sizes: tuple of integers ex: (100,50,30)
        # activation: set string {‘identity’, ‘logistic’, ‘tanh’, ‘relu’}
        # learning_rate: set string {‘constant’, ‘invscaling’, ‘adaptive’}
mlp_model = mlp(hidden_layer_sizes = (300,150,100,50,25), activation = 'relu', learning_rate = 'constant')

# Train the model 
mlp_model.fit(x_tr, y_tr)

# Generate Predictions
mlp_tr = mlp_model.predict(x_tr)
mlp_te = mlp_model.predict(x_te)

# Evaluate
evaluate_regression(y_tr, mlp_tr, y_te, mlp_te)

# Hyperparametric Optimization

In [ ]:
# Define model
model = gbf()

# Define parameter ranges
param_grid = {
              'n_estimators': [100, 150, 200, 250],
              'max_depth': [2,3,4,6],
              'max_features': ['sqrt', 'log2', None]
             }

# Create GridSearchCV object
folds = 5
grid_search = GridSearchCV(model, param_grid, cv=folds)

# Fit the model to the data
grid_search.fit(x_tr, y_tr)

# Print the best parameters
best = grid_search.best_params_
print("Best parameters: ", best)

# Generate Predictions
grid_tr = grid_search.predict(x_tr)
grid_te = grid_search.predict(x_te)

# Evaluate
evaluate_regression(y_tr, grid_tr, y_te, grid_te)

# Final Training

In [ ]:
# After you've selected a model, you must retrain the model on the ENTIRE data set

# Concatenate the training and test sets
X = pd.concat([x_tr, x_te], axis = 0)
X.reset_index(drop=True, inplace=True)

y = pd.concat([y_tr, y_te], axis = 0)
y.reset_index(drop=True, inplace=True)

# Retrain the model on the complete set using the best hyperparameters
final = gbf(n_estimators = 100, max_depth = 4, max_features = 'sqrt')

# Train the model 
final.fit(X, y)

### Generate New Predictions

In [ ]:
# Import data set
path = r"C:\Users\hunte\OneDrive\Documents\Marquette\FINA4075 Intro to Fintech\AI Project\cars_ai_practice_new.csv"
new = pd.read_csv(path)

# Alphebetize the columns
new.sort_index(axis=1, inplace=True)
new.head()

In [ ]:
# Generate predictions and append to the data set
new['pred_price'] = final.predict(new)

# Export the data
path = r"C:\Users\hunte\OneDrive\Documents\Marquette\FINA4075 Intro to Fintech\AI Project\cars_predictions.csv"
new.to_csv(path, index=False)